In [ ]:
import numpy as np
import pandas as pd

import shap
import joblib

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")


In [ ]:
#model = joblib.load("C:/Users/Durosimi/PROJECTS/Parkinson/models/final_motor_updrs_groupcv.pkl")
data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_df.csv")

In [ ]:
data.columns

In [ ]:
data["updrs_lag1"] = data.groupby("subject")["motor_updrs"].shift(1)

In [ ]:
data = data.dropna(subset=["updrs_lag1"]).reset_index(drop=True)

In [ ]:
data["updrs_lag1"].isna().sum()

In [ ]:
subjects = data["subject"]

In [ ]:
features = [
    "updrs_lag1",
    "baseline_updrs",
     "jitter",
    "shimmer",
    "hnr",
    "rpde",
    "ppe",
    "sex"
]

X = data[features]
y = data["delta_motor_updrs"]

### Refit the model

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=0.1))
])

In [ ]:
ridge_pipeline.fit(X, y)

In [ ]:
coef = ridge_pipeline.named_steps["ridge"].coef_

coef_df = pd.DataFrame({
    "features":features,
    "coefficient": coef
}).sort_values("coefficient")

coef_df

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=coef_df,
    x="coefficient",
    y="features"
)
plt.axvline(0, color="black", linestyle="--")
plt.title("Ridge Model Coefficients (Standardized)")
plt.tight_layout()
plt.show()


In [ ]:
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df.sort_values("abs_coef", ascending=False)

In [ ]:
y_pred = ridge_pipeline.predict(X)

print("Prediction std:", np.std(y_pred))
print("Target std:", np.std(y))


In [ ]:
eval_df = data[["subject", "test_time"]].copy()
eval_df["y_true"] = y.values
eval_df["y_pred"] = y_pred


In [ ]:
subject_id = eval_df["subject"].unique()[41]
patient = eval_df[eval_df["subject"] == subject_id]

plt.figure(figsize=(8,4))
plt.plot(patient["test_time"], patient["y_true"], marker="o", label="Actual")
plt.plot(patient["test_time"], patient["y_pred"], marker="x", linestyle="--", label="Predicted")
plt.title(f"UPDRS Progression (Subject {subject_id})")
plt.xlabel("Time")
plt.ylabel("Delta Motor UPDRS")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 1. Check for multicollinearity
import seaborn as sns
import matplotlib.pyplot as plt

# Correlation matrix
corr_matrix = data[['updrs_lag1', 'baseline_updrs', 'jitter', 'shimmer', 'hnr', 'rpde', 'ppe', 'sex']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

# 2. Check if baseline_updrs and updrs_lag1 are highly correlated
print(f"Correlation between baseline_updrs and updrs_lag1: {data['baseline_updrs'].corr(data['updrs_lag1']):.3f}")

# 3. Check coefficient stability with different alpha values
alphas = [0.01, 0.1, 1, 10, 100]
coef_stability = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X, y)
    coef_stability.append(ridge.coef_)

In [ ]:
from sklearn.model_selection import GroupKFold

coefs = []

cv = GroupKFold(n_splits=5)

for train_idx, test_idx in cv.split(X, y, groups=subjects):
    ridge_pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
    coefs.append(
        ridge_pipeline.named_steps["ridge"].coef_
    )

coef_stability = pd.DataFrame(coefs, columns=features)
coef_stability.describe()


### Model predictions were primarily driven by short-term temporal continuity and baseline disease severity, with secondary contributions from voice-derived features. The negative association between baseline severity and progression magnitude reflects regression-to-the-mean effects and is clinically plausible.

In [ ]:
# explainer = shap.TreeExplainer(model.named_steps["model"])
# shap_values = explainer.shap_values(
#     model.named_steps["scaler"].transform(X)
# )

In [ ]:
# shap.summary_plot(
#     shap_values,
#     model.named_steps["scaler"].transform(X),
#     feature_names=features,
#     plot_type="bar"
# )

In [ ]:
# shap.summary_plot(
#     shap_values,
#     model.named_steps["scaler"].transform(X),
#     feature_names=features
# )


In [ ]:
# idx = 3

# shap.force_plot(
#     explainer.expected_value,
#     shap_values[idx],
#     model.named_steps["scaler"].transform(X)[idx],
#     feature_names=features,
#     matplotlib=True
# )


In [ ]:
# for feature in features:
#     shap.dependence_plot(
#         feature,
#         shap_values,
#         model.named_steps["scaler"].transform(X),
#         feature_names=features
#     )


In [ ]:
# df_shap = pd.DataFrame(
#     shap_values,
#     columns=features
# )

# df_shap["sex"] = data["sex"].values

# sns.boxplot(
#     x="sex",
#     y="shimmer",
#     data=df_shap
# )
# plt.title("SHAP impact of shimmer by sex")
# plt.show()


<!-- ### SHAP analysis indicates that voice perturbation measures (jitter, shimmer, RPDE) contribute most strongly to motor UPDRS predictions. Higher jitter and shimmer values generally increase predicted motor impairment, while higher harmonic-to-noise ratio (HNR) is associated with lower predicted severity. Sex has a comparatively small but non-zero contribution. -->